# 06 — Write Once, Deploy to LangChain, CrewAI, AutoGen, and Seven More

**The pitch.** Every agent framework has its own prompt format. LangChain wants messages. CrewAI wants role/goal/backstory. AutoGen wants system_message. Migrating a prompt from one to another means rewriting it by hand. mycontext's `Context` object is framework-agnostic: build once, export to ten frameworks with a single method call — or create full agent objects directly.

**Who this is for:** Developers working across multiple agent frameworks, teams evaluating frameworks, anyone who hates rewriting prompts.

**What you'll learn:**
- All 10 export methods: `.to_openai()`, `.to_langchain()`, `.to_crewai()`, `.to_autogen()`, `.to_anthropic()`, `.to_google()`, `.to_llamaindex()`, `.to_prompt()`, `.to_messages()`, `.to_markdown()`
- Integration helpers: `CrewAIHelper.create_agent()`, `AutoGenHelper.create_assistant()`, `LangChainHelper.to_messages()`, `GoogleADKHelper.create_agent()`
- `auto_integrate(ctx, 'crewai')` — dispatch by framework name string
- Quality equivalence: same prompt, same question, same scores across frameworks
- Optional **live execution** (API key): run **`ctx.execute()`** on several different cognitive templates — not the same `code_reviewer` demo

**Next:** **07** = Executable skills — the SKILL.md that runs, measures, and improves.

## Research grounding

> **Portability-performance:** The same semantic prompt, adapted to each framework's native format, produces statistically equivalent output quality across OpenAI, Anthropic, CrewAI, and AutoGen — measured via `OutputEvaluator` in mycontext's cross-framework evaluation. Framework choice is therefore a deployment decision, not a quality one.

> **Provider-aware rendering:** `Context.assemble_for_model(provider)` applies provider-specific formatting backed by empirical testing: Anthropic performs better with XML delimiters and positive constraint reframes; Gemini with explicit verbosity anchors and trait adjectives on the role; OpenAI with instruction mirroring for long knowledge blocks *(mycontext provider rendering research, 2026-03-16)*.

> **Composable quality (Blueprint):** For extra anti-fluff or tone shaping on any `Context`, `from mycontext import fragments` and e.g. `fragments.anti_fluff.apply(ctx)` — small blueprint-oriented helpers you can chain outside of a full template.

## The framework format zoo

| Framework | Format mycontext produces |
|-----------|---------------------------|
| OpenAI | `{messages: [{role, content}], model, temperature}` |
| Anthropic | `{system: str, messages: [...]}` |
| Google Gemini | `{contents: [...], generation_config: {...}}` |
| LangChain | `{system_message: str, human_message: str}` |
| LlamaIndex | `{system_prompt: str, query_instruction: str}` |
| CrewAI | `{role: str, goal: str, backstory: str, expected_output: str}` |
| AutoGen | `{system_message: str, name: str, ...}` |
| DSPy | Signature string |
| Semantic Kernel | Prompt template |
| Google ADK | Instruction string + agent object |

## Install and setup

In [1]:
# %pip install -q -U "mycontext-ai>=0.11.1"
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s, raw=True)


def _preview_strings(d: dict, max_len: int = 800) -> dict:
    """Truncate long string values so framework JSON is readable in the notebook."""
    out = {}
    for k, v in d.items():
        if isinstance(v, str) and len(v) > max_len:
            out[k] = v[:max_len] + f"\n\n… [{len(v)} chars total — see full `ctx.to_*()`]"
        else:
            out[k] = v
    return out


import mycontext

md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")


**mycontext-ai** `0.11.0`

## Step 1 — Build one Context (build once)

In [2]:
from mycontext.intelligence import get_pattern_class

# Use code_reviewer as our example — concrete, well-parameterized
# get_pattern_class returns the class; build_context is an instance method.
CodeReviewerCls = get_pattern_class("code_reviewer")

sample_code = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)
"""

ctx = CodeReviewerCls().build_context(
    question="Review this code for security vulnerabilities",
    code=sample_code,
    language="python",
)

md(f"Context built. Template: `{ctx.metadata.get('template', 'code_reviewer')}`")
md(f"Assembled prompt length: {len(ctx.assemble())} chars")


Context built. Template: `code_reviewer`

Assembled prompt length: 5860 chars

## Step 2 — All 10 export formats (deploy anywhere)

In [3]:
import json
# OpenAI
openai_fmt = ctx.to_openai()
md("=== to_openai() ===")
_sj = json.dumps(openai_fmt, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== to_openai() ===

```json
{
  "messages": [
    {
      "role": "system",
      "content": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critique\n  - risk assessment with impact, likelihood, and blast radius for each finding\n  - concrete code examples for fixes\n  - failure scenarios for each finding\n\nMust NOT include:\n  - style, formatting, or naming convention feedback\n  - vague generalities without specific locations\n  - criticisms without constructive solutions\n  - invented findings when no real issues exist\n\nSELF-VERIFICATION \u2014 before finalizing, confirm:\n  - Did I check for OWASP top 10 vulnerabilities?\n  - Are my suggestions actionable with specific line references?\n\nReview this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly"
    }
  ],
  "temperature": 0.7,
  "max_tokens": 4096
}
```

In [4]:
import json
# Anthropic
anthropic_fmt = ctx.to_anthropic()
md("=== to_anthropic() ===")
_sj = json.dumps(anthropic_fmt, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== to_anthropic() ===

```json
{
  "system": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critique\n  - risk assessment with impact, likelihood, and blast radius for each finding\n  - concrete code examples for fixes\n  - failure scenarios for each finding\n\nMust NOT include:\n  - style, formatting, or naming convention feedback\n  - vague generalities without specific locations\n  - criticisms without constructive solutions\n  - invented findings when no real issues exist\n\nSELF-VERIFICATION \u2014 before finalizing, confirm:\n  - Did I check for OWASP top 10 vulnerabilities?\n  - Are my suggestions actionable with specific line references?\n\nReview this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "messages": [],
  "max_tokens": 4096
}
```

In [5]:
import json
# CrewAI
crewai_fmt = ctx.to_crewai()
md("=== to_crewai() ===")
_sj = json.dumps(crewai_fmt, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== to_crewai() ===

```json
{
  "role": "Senior Software Engineer and Code Review Expert",
  "goal": "Review this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "backstory": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented",
  "context": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critique\n  - risk assessment with impact, likelihood, and blast radius for each finding\n  - concrete code examples for fixes\n  - failure scenarios for each finding\n\nMust NOT include:\n  - style, formatting, or naming convention feedback\n  - vague generalities without specific locations\n  - criticisms without constructive solutions\n  - invented findings when no real issues exist\n\nSELF-VERIFICATION \u2014 before finalizing, confirm:\n  - Did I check for OWASP top 10 vulnerabilities?\n  - Are my suggestions actionable with specific line references?\n\nReview this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "expected_output": "Output must include: orientation phase showing understanding of the code before critique, risk assessment with impact, likelihood, and blast radius for each finding, concrete code examples for fixes, failure scenarios for each finding",
  "tools": [],
  "verbose": true
}
```

In [6]:
import json
# AutoGen
autogen_fmt = ctx.to_autogen()
md("=== to_autogen() ===")
_sj = json.dumps(autogen_fmt, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== to_autogen() ===

```json
{
  "system_message": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critique\n  - risk assessment with impact, likelihood, and blast radius for each finding\n  - concrete code examples for fixes\n  - failure scenarios for each finding\n\nMust NOT include:\n  - style, formatting, or naming convention feedback\n  - vague generalities without specific locations\n  - criticisms without constructive solutions\n  - invented findings when no real issues exist\n\nSELF-VERIFICATION \u2014 before finalizing, confirm:\n  - Did I check for OWASP top 10 vulnerabilities?\n  - Are my suggestions actionable with specific line references?\n\nReview this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "description": "Review this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "max_consecutive_auto_reply": 10,
  "human_input_mode": "NEVER",
  "code_execution_config": false
}
```

In [7]:
import json
# LangChain, Google Gemini, LlamaIndex
md("=== to_langchain() ===")
_sj = json.dumps(ctx.to_langchain(), indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")

md("\n=== to_google() ===")
_sj = json.dumps(ctx.to_google(), indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")

md("\n=== to_llamaindex() ===")
_sj = json.dumps(ctx.to_llamaindex(), indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== to_langchain() ===

```json
{
  "system_message": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critique\n  - risk assessment with impact, likelihood, and blast radius for each finding\n  - concrete code examples for fixes\n  - failure scenarios for each finding\n\nMust NOT include:\n  - style, formatting, or naming convention feedback\n  - vague generalities without specific locations\n  - criticisms without constructive solutions\n  - invented findings when no real issues exist\n\nSELF-VERIFICATION \u2014 before finalizing, confirm:\n  - Did I check for OWASP top 10 vulnerabilities?\n  - Are my suggestions actionable with specific line references?\n\nReview this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "context": {
    "guidance": {
      "role": "Senior Software Engineer and Code Review Expert",
      "rules": [
        "Understand before criticizing \u2014 state what the code does before finding issues",
        "Focus on what linters cannot catch \u2014 skip style, formatting, and naming",
        "Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius",
        "Think about failure modes \u2014 what breaks in production, not just what looks wrong",
        "Provide concrete fixes \u2014 every finding must include a working code example",
        "Acknowledge strengths \u2014 note what the code does well"
      ],
      "style": "risk-aware, specific, constructive, failure-mode-oriented",
      "expertise": null,
      "goal": null,
      "persona_scope": null
    },
    "directive": {
      "content": "Review this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
      "priority": 5,
      "constraints": null,
      "tags": null
    },
    "constraints": {
      "must_include": [
        "orientation phase showing understanding of the code before critique",
        "risk assessment with impact, likelihood, and blast radius for each finding",
   ...
```


=== to_google() ===

```json
{
  "contents": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critique\n  - risk assessment with impact, likelihood, and blast radius for each finding\n  - concrete code examples for fixes\n  - failure scenarios for each finding\n\nMust NOT include:\n  - style, formatting, or naming convention feedback\n  - vague generalities without specific locations\n  - criticisms without constructive solutions\n  - invented findings when no real issues exist\n\nSELF-VERIFICATION \u2014 before finalizing, confirm:\n  - Did I check for OWASP top 10 vulnerabilities?\n  - Are my suggestions actionable with specific line references?\n\nReview this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "generation_config": {
    "temperature": 0.7,
    "max_output_tokens": 4096
  }
}
```


=== to_llamaindex() ===

```json
{
  "template": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critique\n  - risk assessment with impact, likelihood, and blast radius for each finding\n  - concrete code examples for fixes\n  - failure scenarios for each finding\n\nMust NOT include:\n  - style, formatting, or naming convention feedback\n  - vague generalities without specific locations\n  - criticisms without constructive solutions\n  - invented findings when no real issues exist\n\nSELF-VERIFICATION \u2014 before finalizing, confirm:\n  - Did I check for OWASP top 10 vulnerabilities?\n  - Are my suggestions actionable with specific line references?\n\nReview this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "system_prompt": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\n5. Provide concrete fixes \u2014 every finding must include a working code example\n6. Acknowledge strengths \u2014 note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented",
  "query_instruction": "Review this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\n\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\n\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\n\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\n\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\n\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don't test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\n\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\n\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\n\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\n\n---\n\n## 3. ASSESS\n\nFor each finding from the ANALYZE phase, assign a risk profile:\n\n| Finding | Impact | Likelihood | Blast Radius | Priority |\n|---------|--------|------------|--------------|----------|\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\n\n**Impact**: What is the worst outcome if this issue manifests in production?\n**Likelihood**: How probable is this failure path in real-world usage?\n**Blast Radius**: If it fails, how much of the system is affected?\n**Priority**: Based on the combination of the three factors above.\n\n---\n\n## 4. STRENGTHS\n\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\n- [Strength 1]\n- [Strength 2]\n\n---\n\n## 5. RECOMMEND\n\nExactly 3 priority actions, ordered by risk (highest first). For each:\n\n### Action 1: [One-sentence finding]\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\n- **Fix**:\n```python\n# Before\n[problematic code]\n\n# After\n[fixed code]\n```\n- **Why this fix**: [Brief explanation of the approach]\n\n### Action 2: [One-sentence finding]\n[Same format]\n\n### Action 3: [One-sentence finding]\n[Same format]\n\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\n\n---\n\n**REQUIREMENTS**:\n- Reference exact locations (function names, line numbers) for every finding\n- Every finding MUST include a failure scenario (what breaks, not just what's wrong)\n- Every recommendation MUST include a working code fix\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\n- If a requested focus area has no issues, say so explicitly",
  "context_str": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing \u2014 state what the code does before finding issues\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\n4. Think about failure modes \u2014 what breaks in production, not just what...
```

## Step 3 — Integration helpers: create full agent objects

In [8]:
import json
from mycontext.integrations.helpers import CrewAIHelper, AutoGenHelper, LangChainHelper

# What you actually want to read: the dict mycontext passes into CrewAI's Agent(...)
crewai_payload = ctx.to_crewai()
md("=== CrewAI payload (`ctx.to_crewai()`) — role / goal / backstory / expected_output ===")
_sj = json.dumps(_preview_strings(crewai_payload), indent=2, ensure_ascii=False)
md("```json\n" + _sj + "\n```")

md(
    "**Note:** `CrewAIHelper.create_agent()` returns a live `crewai.Agent` object, not a dict. "
    "`json.dumps(agent)` only stringifies Python's `repr()` — you get `id=UUID(...)` noise. "
    "Use `ctx.to_crewai()` for the prompt payload, or the runtime summary below."
)

md("=== `CrewAIHelper.create_agent()` — runtime object summary (JSON-safe) ===")
try:
    crewai_agent = CrewAIHelper.create_agent(ctx, name="SecurityReviewer")
    summary = {
        "type": type(crewai_agent).__name__,
        "role": getattr(crewai_agent, "role", None),
        "goal": getattr(crewai_agent, "goal", None),
        "backstory": getattr(crewai_agent, "backstory", None),
        "verbose": getattr(crewai_agent, "verbose", None),
        "tools": [getattr(t, "name", str(t)) for t in (getattr(crewai_agent, "tools", None) or [])],
    }
    _sj2 = json.dumps(_preview_strings(summary, max_len=800), indent=2, ensure_ascii=False)
    md("```json\n" + _sj2 + "\n```")
except Exception as e:
    md(f"Could not instantiate CrewAI `Agent` (often missing `OPENAI_API_KEY` for default LLM): `{e}`")
    md("Payload above is still valid — pass it to `Agent(...)` yourself with your LLM config.")


=== CrewAI payload (`ctx.to_crewai()`) — role / goal / backstory / expected_output ===

```json
{
  "role": "Senior Software Engineer and Code Review Expert",
  "goal": "Review this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input → processing → output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus are\n\n… [4521 chars total — see full `ctx.to_*()`]",
  "backstory": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing — state what the code does before finding issues\n2. Focus on what linters cannot catch — skip style, formatting, and naming\n3. Assess risk, not just severity — every finding needs impact, likelihood, and blast radius\n4. Think about failure modes — what breaks in production, not just what looks wrong\n5. Provide concrete fixes — every finding must include a working code example\n6. Acknowledge strengths — note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented",
  "context": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing — state what the code does before finding issues\n2. Focus on what linters cannot catch — skip style, formatting, and naming\n3. Assess risk, not just severity — every finding needs impact, likelihood, and blast radius\n4. Think about failure modes — what breaks in production, not just what looks wrong\n5. Provide concrete fixes — every finding must include a working code example\n6. Acknowledge strengths — note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critiq\n\n… [5860 chars total — see full `ctx.to_*()`]",
  "expected_output": "Output must include: orientation phase showing understanding of the code before critique, risk assessment with impact, likelihood, and blast radius for each finding, concrete code examples for fixes, failure scenarios for each finding",
  "tools": [],
  "verbose": true
}
```

**Note:** `CrewAIHelper.create_agent()` returns a live `crewai.Agent` object, not a dict. `json.dumps(agent)` only stringifies Python's `repr()` — you get `id=UUID(...)` noise. Use `ctx.to_crewai()` for the prompt payload, or the runtime summary below.

=== `CrewAIHelper.create_agent()` — runtime object summary (JSON-safe) ===

```json
{
  "type": "Agent",
  "role": "Senior Software Engineer and Code Review Expert",
  "goal": "Review this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input → processing → output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus are\n\n… [4521 chars total — see full `ctx.to_*()`]",
  "backstory": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing — state what the code does before finding issues\n2. Focus on what linters cannot catch — skip style, formatting, and naming\n3. Assess risk, not just severity — every finding needs impact, likelihood, and blast radius\n4. Think about failure modes — what breaks in production, not just what looks wrong\n5. Provide concrete fixes — every finding must include a working code example\n6. Acknowledge strengths — note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented",
  "verbose": true,
  "tools": []
}
```

In [9]:
import json

try:
    _preview_strings
except NameError:

    def _preview_strings(d: dict, max_len: int = 800) -> dict:
        out = {}
        for k, v in d.items():
            if isinstance(v, str) and len(v) > max_len:
                out[k] = v[:max_len] + f"\n\n… [{len(v)} chars total — see full `ctx.to_*()`]"
            else:
                out[k] = v
        return out

from mycontext.integrations.helpers import AutoGenHelper

autogen_payload = ctx.to_autogen()
md("=== AutoGen payload (`ctx.to_autogen()`) ===")
_sj = json.dumps(_preview_strings(autogen_payload, max_len=1200), indent=2, ensure_ascii=False)
md("```json\n" + _sj + "\n```")

md("=== `AutoGenHelper.create_assistant()` — runtime summary ===")
try:
    autogen_agent = AutoGenHelper.create_assistant(ctx, name="SecurityReviewAgent")
    ag_summary = {
        "type": type(autogen_agent).__name__,
        "name": getattr(autogen_agent, "name", None),
        "system_message": getattr(autogen_agent, "system_message", None),
    }
    _sj2 = json.dumps(_preview_strings(ag_summary, max_len=1200), indent=2, ensure_ascii=False)
    md("```json\n" + _sj2 + "\n```")
except Exception as e:
    md(f"Could not instantiate AutoGen assistant: `{e}`")


=== AutoGen payload (`ctx.to_autogen()`) ===

```json
{
  "system_message": "You are Senior Software Engineer and Code Review Expert.\n\nFollow these rules:\n1. Understand before criticizing — state what the code does before finding issues\n2. Focus on what linters cannot catch — skip style, formatting, and naming\n3. Assess risk, not just severity — every finding needs impact, likelihood, and blast radius\n4. Think about failure modes — what breaks in production, not just what looks wrong\n5. Provide concrete fixes — every finding must include a working code example\n6. Acknowledge strengths — note what the code does well\n\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\n\nCONSTRAINTS:\n\nStyle: Use markdown formatting with code blocks and risk assessment tables\n\nMust include:\n  - orientation phase showing understanding of the code before critique\n  - risk assessment with impact, likelihood, and blast radius for each finding\n  - concrete code examples for fixes\n  - failure scenarios for each finding\n\nMust NOT include:\n  - style, formatting, or naming convention feedback\n  - vague generalities without specific locations\n  - criticisms without constructive solutions\n  - invented findings when no real issues exist\n\nSELF-VERIFICATION — befor\n\n… [5860 chars total — see full `ctx.to_*()`]",
  "description": "Review this python code using a structured cognitive process.\n\n**CODE TO REVIEW**:\n```python\n\ndef get_user(user_id):\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\n    return db.execute(query)\n\n```\n\n\n\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\n\n---\n\n## 1. ORIENT\n\nBefore finding any issues, build a mental model of this code:\n\n- **Purpose**: What does this code do? (1-2 sentences)\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\n- **Data flow**: How does data move through this code? (input → processing → output)\n\n---\n\n## 2. ANALYZE\n\nExamine the code across the requested focus areas. For each finding, use this format:\n\n> **[Dimension] Finding: [One-sentence description]**\n> - **Location**: [Function name, line number, or code reference]\n> - **What's wrong**: [Specific technical explanation]\n> - **What fails**: [The failure scenario — what breaks and under what conditions]\n\nDimensions to check (based on focus areas):\n\n**Correctness** — Logic errors, edge cases, off-by-one e\n\n… [4521 chars total — see full `ctx.to_*()`]",
  "max_consecutive_auto_reply": 10,
  "human_input_mode": "NEVER",
  "code_execution_config": false
}
```

=== `AutoGenHelper.create_assistant()` — runtime summary ===

Could not instantiate AutoGen assistant: `AutoGen is not installed. Install with: pip install pyautogen`

In [10]:
import json
from mycontext.integrations.helpers import LangChainHelper

# LangChain: prompt messages
lc_msgs = LangChainHelper.to_messages(ctx, user_message="Review the code above")
md("=== LangChainHelper.to_messages() ===")
_sj = json.dumps(lc_msgs, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


=== LangChainHelper.to_messages() ===

```json
[
  "content='You are Senior Software Engineer and Code Review Expert.\\n\\nFollow these rules:\\n1. Understand before criticizing \u2014 state what the code does before finding issues\\n2. Focus on what linters cannot catch \u2014 skip style, formatting, and naming\\n3. Assess risk, not just severity \u2014 every finding needs impact, likelihood, and blast radius\\n4. Think about failure modes \u2014 what breaks in production, not just what looks wrong\\n5. Provide concrete fixes \u2014 every finding must include a working code example\\n6. Acknowledge strengths \u2014 note what the code does well\\n\\nCommunication style: risk-aware, specific, constructive, failure-mode-oriented\\n\\nCONSTRAINTS:\\n\\nStyle: Use markdown formatting with code blocks and risk assessment tables\\n\\nMust include:\\n  - orientation phase showing understanding of the code before critique\\n  - risk assessment with impact, likelihood, and blast radius for each finding\\n  - concrete code examples for fixes\\n  - failure scenarios for each finding\\n\\nMust NOT include:\\n  - style, formatting, or naming convention feedback\\n  - vague generalities without specific locations\\n  - criticisms without constructive solutions\\n  - invented findings when no real issues exist\\n\\nSELF-VERIFICATION \u2014 before finalizing, confirm:\\n  - Did I check for OWASP top 10 vulnerabilities?\\n  - Are my suggestions actionable with specific line references?\\n\\nReview this python code using a structured cognitive process.\\n\\n**CODE TO REVIEW**:\\n```python\\n\\ndef get_user(user_id):\\n    query = f\"SELECT * FROM users WHERE id = {{user_id}}\"\\n    return db.execute(query)\\n\\n```\\n\\n\\n\\n**FOCUS AREAS**: correctness, security, performance, design, resilience, testing, maintainability\\n\\n---\\n\\n## 1. ORIENT\\n\\nBefore finding any issues, build a mental model of this code:\\n\\n- **Purpose**: What does this code do? (1-2 sentences)\\n- **Key abstractions**: What are the main data structures, functions, or classes and how do they relate?\\n- **Assumptions**: What does this code assume about its inputs, environment, or dependencies?\\n- **Data flow**: How does data move through this code? (input \u2192 processing \u2192 output)\\n\\n---\\n\\n## 2. ANALYZE\\n\\nExamine the code across the requested focus areas. For each finding, use this format:\\n\\n> **[Dimension] Finding: [One-sentence description]**\\n> - **Location**: [Function name, line number, or code reference]\\n> - **What\\'s wrong**: [Specific technical explanation]\\n> - **What fails**: [The failure scenario \u2014 what breaks and under what conditions]\\n\\nDimensions to check (based on focus areas):\\n\\n**Correctness** \u2014 Logic errors, edge cases, off-by-one errors, null/undefined handling, boundary conditions, race conditions. Does the code actually do what it claims to?\\n\\n**Security** \u2014 Injection vectors (SQL, XSS, command), authentication/authorization flaws, data exposure, hardcoded secrets, insecure defaults. Can this be exploited?\\n\\n**Performance** \u2014 Algorithmic complexity, N+1 queries, unnecessary allocations, resource leaks, missing connection pooling, unbounded growth. Will this scale?\\n\\n**Design** \u2014 SOLID violations, inappropriate coupling, wrong abstraction level, over-engineering, missing separation of concerns. Is this well-structured for future change?\\n\\n**Resilience** \u2014 Missing error handling, unhandled exception paths, no timeouts on external calls, no retry logic, silent failures, cascade failure risk. What happens when things go wrong?\\n\\n**Testing** \u2014 Missing test coverage for critical paths, assertions that don\\'t test meaningful behavior, untested error paths, missing edge case tests. Will the test suite catch regressions?\\n\\n**Maintainability** \u2014 Unnecessary complexity, duplicated logic, functions doing too many things, unclear control flow. Can the next developer understand and safely modify this?\\n\\n**IMPORTANT**: Do NOT comment on style, formatting, naming conventions, import ordering, or docstring presence. These are linter concerns. Spend your analysis on dimensions above that linters cannot catch.\\n\\nOnly analyze dimensions listed in the focus areas. If a dimension has no findings, state \"No issues found\" \u2014 do not force findings.\\n\\n---\\n\\n## 3. ASSESS\\n\\nFor each finding from the ANALYZE phase, assign a risk profile:\\n\\n| Finding | Impact | Likelihood | Blast Radius | Priority |\\n|---------|--------|------------|--------------|----------|\\n| [Finding summary] | [Data loss / System crash / Degradation / Cosmetic] | [Always triggered / Common path / Edge case / Theoretical] | [Full system / Single service / One module / Local] | [Fix before merge / Fix soon / Track for later] |\\n\\n**Impact**: What is the worst outcome if this issue manifests in production?\\n**Likelihood**: How probable is this failure path in real-world usage?\\n**Blast Radius**: If it fails, how much of the system is affected?\\n**Priority**: Based on the combination of the three factors above.\\n\\n---\\n\\n## 4. STRENGTHS\\n\\nWhat this code does well (be specific \u2014 reference actual patterns, decisions, or techniques):\\n- [Strength 1]\\n- [Strength 2]\\n\\n---\\n\\n## 5. RECOMMEND\\n\\nExactly 3 priority actions, ordered by risk (highest first). For each:\\n\\n### Action 1: [One-sentence finding]\\n- **Risk**: [Impact + Likelihood + Blast Radius summary from ASSESS]\\n- **Fix**:\\n```python\\n# Before\\n[problematic code]\\n\\n# After\\n[fixed code]\\n```\\n- **Why this fix**: [Brief explanation of the approach]\\n\\n### Action 2: [One-sentence finding]\\n[Same format]\\n\\n### Action 3: [One-sentence finding]\\n[Same format]\\n\\nIf fewer than 3 issues were found, include only the issues that exist \u2014 do not invent findings to fill the slots.\\n\\n---\\n\\n**REQUIREMENTS**:\\n- Reference exact locations (function names, line numbers) for every finding\\n- Every finding MUST include a failure scenario (what breaks, not just what\\'s wrong)\\n- Every recommendation MUST include a working code fix\\n- Do NOT include style, formatting, or naming feedback \u2014 linters handle those\\n- If a requested focus area has no issues, say so explicitly' additional_kwargs={} response_metadata={}",
  "content='Review the code above' additional_kwargs={} response_metadata={}"
]
```

## Step 4 — `auto_integrate`: dispatch by name string

In [18]:
!uv add pyautogen

Resolved 323 packages in 3ms
Checked 317 packages in 50ms


In [19]:
from mycontext.integrations.helpers import auto_integrate

# Same ctx, different framework — by string dispatch (each call isolated; missing deps won't stop the rest)
for framework in ["crewai", "autogen", "langchain"]:
    md(f"auto_integrate(ctx, '{framework}'):")
    try:
        result = auto_integrate(ctx, framework)
        if isinstance(result, dict):
            md(f"  Keys: {list(result.keys())}")
        else:
            md(f"  Type: {type(result).__name__}")
    except ImportError as e:
        md(f"  Skipped (optional dependency): `{e}`")
    except Exception as e:
        md(f"  Error: `{e}`")
    md("---")


auto_integrate(ctx, 'crewai'):

  Type: Agent

---

auto_integrate(ctx, 'autogen'):

  Skipped (optional dependency): `AutoGen is not installed. Install with: pip install pyautogen`

---

auto_integrate(ctx, 'langchain'):

  Type: list

---

## Step 5 — Quality equivalence: same score, any framework

In [12]:
from mycontext.intelligence import QualityMetrics

# The Context quality score is framework-independent
# — it scores the assembled prompt, not the export format
qm = QualityMetrics(mode="heuristic")
score = qm.evaluate(ctx)

md(f"Prompt quality score: {score.overall * 100:.0f}/100")
md("(This score is the same regardless of which framework you export to)")
md("\nOne Context → ten frameworks → identical prompt quality.")
md("Migrate from LangChain to CrewAI without touching the prompt.")


Prompt quality score: 84/100

(This score is the same regardless of which framework you export to)


One Context → ten frameworks → identical prompt quality.

Migrate from LangChain to CrewAI without touching the prompt.

## Step 6 — Live execution: different templates, one simple API

Requires **`OPENAI_API_KEY`**. This does **not** reuse `code_reviewer`: we build four **other** patterns, then call **`context.execute(provider="openai", user=…)`** on each — the same one-liner you would use in an app.

Templates here: **`data_analyzer`**, **`risk_assessor`**, **`question_analyzer`**, **`synthesis_builder`** — each gets its own short user turn so answers stay compact.

In [17]:
from mycontext.intelligence import get_pattern_class


def build_exec_contexts():
    """Four distinct templates (not code_reviewer) with minimal valid `build_context` args."""
    g = get_pattern_class
    return [
        (
            "data_analyzer",
            g("data_analyzer")().build_context(
                data_description=(
                    "Daily active users flat for 3 weeks while new signups rose 12% week over week. "
                    "No major product or pricing changes."
                ),
                goal="Surface plausible explanations and what data to check next",
                intent="summary",
            ),
            "In under 100 words: your top 2 hypotheses for the DAU / signup mismatch and one metric to validate each.",
        ),
        (
            "risk_assessor",
            g("risk_assessor")().build_context(
                decision="Roll out MFA to admins first, then all users within 90 days.",
                context="B2B SaaS, ~4k organizations, SOC2 audit in flight, mobile apps use refresh tokens.",
                depth="basic",
            ),
            "In under 100 words: name the single highest residual risk after this rollout plan and one mitigation.",
        ),
        (
            "question_analyzer",
            g("question_analyzer")().build_context(
                question=(
                    "Should we rebuild the billing service on a new stack or incrementally refactor the legacy module?"
                ),
                depth="comprehensive",
            ),
            "In under 100 words: list 3 sub-questions the leadership team should answer before deciding.",
        ),
        (
            "synthesis_builder",
            g("synthesis_builder")().build_context(
                sources=[
                    "Engineering: API latency p95 improved 18% after cache rollout.",
                    "Support: ticket volume up 22%; top theme is billing preview accuracy.",
                    "Sales: two enterprise deals stalled on SSO integration timeline.",
                ],
                goal="One coherent exec read for Monday staff meeting",
                context="Conflicting signals between reliability wins and customer friction",
            ),
            "In under 120 words: synthesize into 3 bullets — theme, tension, and one recommended focus for the week.",
        ),
    ]


if not OPENAI_API_KEY:
    md(
        "**Skipping live execution** — set `OPENAI_API_KEY` in your environment or `content-series/.env`, "
        "then re-run from the setup cell."
    )
else:
    rows = build_exec_contexts()
    md("Each block is **`template_name` → `Context.execute()`** (same API for every template).")
    for name, exec_ctx, user_turn in rows:
        md(f"### `{name}`")
        try:
            out = exec_ctx.execute(
                provider="openai",
                user=user_turn,
                api_key=OPENAI_API_KEY,
                model="gpt-4o-mini",
                temperature=0.3,
            )
            md(out.response)
            md(f"*tokens≈{out.tokens_used} · model={out.model}*")
        except Exception as e:
            md(f"`execute` failed: `{e}`")
        md("---")


Each block is **`template_name` → `Context.execute()`** (same API for every template).

### `data_analyzer`

**Hypothesis 1**: User engagement may be declining despite new signups, leading to flat DAU.  
- **Validation Metric**: Track user retention rates over the same period to assess how many new signups remain active.

**Hypothesis 2**: New signups could be driven by a specific marketing campaign that does not resonate with existing users.  
- **Validation Metric**: Analyze the source of new signups to determine if there are significant differences in engagement levels between users acquired through different channels.

*tokens≈799 · model=gpt-4o-mini*

---

...

---

### `synthesis_builder`

- **Theme**: While the engineering team achieved an 18% improvement in API latency through cache rollout, customer support is experiencing a 22% increase in ticket volume, primarily driven by issues related to billing preview accuracy.

- **Tension**: The reliability gains in engineering contrast sharply with customer friction, as evidenced by stalled enterprise deals due to delays in SSO integration, highlighting a disconnect between technical improvements and customer satisfaction.

- **Recommended Focus**: Prioritize enhancing the billing preview accuracy and expedite the SSO integration process to address customer concerns and reduce ticket volume, ensuring that reliability improvements translate into a better customer experience.

*tokens≈782 · model=gpt-4o-mini*

---

## Summary

| What you learned | API | Key? |
|-----------------|-----|------|
| OpenAI messages format | `ctx.to_openai()` | No |
| Anthropic format | `ctx.to_anthropic()` | No |
| CrewAI agent config | `ctx.to_crewai()` | No |
| AutoGen config | `ctx.to_autogen()` | No |
| LangChain, Google, LlamaIndex | `ctx.to_langchain()`, `.to_google()`, `.to_llamaindex()` | No |
| Full agent creation | `CrewAIHelper.create_agent(ctx)`, `AutoGenHelper.create_assistant(ctx)` | No |
| String dispatch | `auto_integrate(ctx, 'crewai')` | No |
| Live run (4 templates) | Step 6 — `execute` on `data_analyzer`, `risk_assessor`, … | Yes (OpenAI) |

**Next notebook:** [07_executable_skills.ipynb](07_executable_skills.ipynb) — the SKILL.md format that executes, quality-gates, and self-improves.